# joint_inv Notebook UI

`sample/joint_inv/joint_inv_minimal.in` をベースに `joint_inv` の入力ファイルを編集し、実行コマンド確認と結果可視化を行う Notebook です。

- 既定の作業ディレクトリ: `sample/joint_inv`
- 既定のテンプレート: `joint_inv_minimal.in`
- 保存先: `joint_inv_ui.in`
- `joint_inv` の main parameter file、`recv_func.in`、`disper.in` はロード後にエディタ上で直接編集できます。
- burn-in 中は最終 posterior 用の `*.ppd`, `*.mean`, `syn_*`, `*_sigma` を読みません。Notebook でも burn-in progress と post-burn posterior を分けて表示します.


In [1]:
from __future__ import annotations

import contextlib
import html
import io
from collections import deque
import os
import re
import shutil
import subprocess
import sys
import threading
import time
import traceback
from pathlib import Path

import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "util" / "sub" / "filoplot.py").exists() and (candidate / "sample" / "joint_inv").exists():
            return candidate
    raise FileNotFoundError("Could not locate the SEIS_FILO repository root.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from util.sub.filoplot import InvResult

DEFAULT_WORKDIR = REPO_ROOT / "sample" / "joint_inv"
DEFAULT_TEMPLATE = DEFAULT_WORKDIR / "joint_inv_minimal.in"
DEFAULT_OUTPUT = DEFAULT_WORKDIR / "joint_inv_ui.in"
DEFAULT_BINARY = REPO_ROOT / "bin" / "joint_inv"



In [2]:
PARAM_SPECS = [
    ("n_iter", "int", 200, 1, 10000000, 1, "Total iterations"),
    ("n_burn", "int", 100, 0, 10000000, 1, "Burn-in iterations"),
    ("n_corr", "int", 10, 1, 1000000, 1, "Thinning interval"),
    ("n_chain", "int", 5, 1, 4096, 1, "Chains per process"),
    ("n_cool", "int", 1, 1, 4096, 1, "Cold chains per process"),
    ("temp_high", "float", 30.0, 0.1, 1000000.0, 0.1, "Highest temperature"),
    ("k_min", "int", 1, 1, 512, 1, "Min layer count"),
    ("k_max", "int", 21, 2, 512, 1, "Max layer count"),
    ("vs_min", "float", 1.0, 0.0, 20.0, 0.1, "Min Vs [km/s]"),
    ("vs_max", "float", 5.0, 0.0, 20.0, 0.1, "Max Vs [km/s]"),
    ("z_min", "float", 2.0, 0.0, 500.0, 0.1, "Min interface depth [km]"),
    ("z_max", "float", 23.0, 0.0, 1000.0, 0.1, "Max interface depth [km]"),
    ("solve_vp", "bool", False, None, None, None, "Solve Vp"),
    ("solve_rf_sig", "bool", True, None, None, None, "Solve RF sigma"),
    ("solve_disper_sig", "bool", True, None, None, None, "Solve dispersion sigma"),
    ("is_sphere", "bool", False, None, None, None, "Apply Earth flattening"),
    ("is_ocean", "bool", True, None, None, None, "Include ocean layer"),
]

LINE_PATTERN = re.compile(r"^(?P<prefix>\s*(?P<key>[A-Za-z0-9_]+)\s*=\s*)(?P<value>[^#\n]*?)(?P<suffix>\s*(?:#.*)?)$")


def parse_bool(text: str) -> bool:
    value = text.strip().lower()
    return value in {"t", ".true.", "true", "1", "yes", "y"}


def format_value(kind: str, value):
    if kind == "bool":
        return "T" if bool(value) else "F"
    if kind == "int":
        return str(int(value))
    if kind == "float":
        return str(float(value))
    return str(value)


def coerce_value(kind: str, text: str):
    raw = text.strip()
    if kind == "bool":
        return parse_bool(raw)
    if kind == "int":
        return int(float(raw))
    if kind == "float":
        return float(raw)
    return raw


def read_param_lines(param_path: Path) -> list[str]:
    return param_path.read_text(encoding="utf-8").splitlines(keepends=True)


def parse_param_text(text: str) -> dict[str, str]:
    values = {}
    for line in text.splitlines():
        match = LINE_PATTERN.match(line.rstrip("\n"))
        if match:
            values[match.group("key")] = match.group("value").strip()
    return values


def read_param_map(param_path: Path) -> dict[str, str]:
    return parse_param_text(param_path.read_text(encoding="utf-8"))


def update_param_lines(lines: list[str], updates: dict[str, str]) -> list[str]:
    updated_lines = []
    seen = set()
    for line in lines:
        stripped = line.rstrip("\n")
        match = LINE_PATTERN.match(stripped)
        if match and match.group("key") in updates:
            key = match.group("key")
            seen.add(key)
            new_line = f"{match.group('prefix')}{updates[key]}{match.group('suffix')}"
            updated_lines.append(new_line + ("\n" if line.endswith("\n") else ""))
        else:
            updated_lines.append(line)
    missing = [key for key in updates if key not in seen]
    if missing:
        raise KeyError(f"Keys not found in template: {missing}")
    return updated_lines


def count_blocks(summary_path: Path) -> int:
    count = 0
    for raw_line in summary_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.split("#", 1)[0].strip()
        if not line:
            continue
        count = int(line)
        break
    return count


@contextlib.contextmanager
def pushd(path: Path):
    current = Path.cwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(current)


def format_timestamp(epoch: float | None) -> str:
    if epoch is None:
        return "N/A"
    return time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(epoch))


def resolve_log_path(workdir: Path) -> Path:
    if current_log_file is not None:
        try:
            current_log_file.relative_to(workdir)
            return current_log_file
        except ValueError:
            pass
    return workdir / "joint_inv_ui.log"


In [3]:
workdir_widget = widgets.Text(value=str(DEFAULT_WORKDIR), description="Workdir", layout=widgets.Layout(width="700px"))
template_widget = widgets.Text(value=str(DEFAULT_TEMPLATE), description="Template", layout=widgets.Layout(width="700px"))
output_widget = widgets.Text(value=str(DEFAULT_OUTPUT), description="Output", layout=widgets.Layout(width="700px"))
mpi_np_widget = widgets.BoundedIntText(value=1, min=1, max=4096, step=1, description="MPI np")
curve_widget = widgets.BoundedIntText(value=1, min=1, max=32, step=1, description="Curve ID")
trace_widget = widgets.BoundedIntText(value=1, min=1, max=32, step=1, description="Trace ID")
interval_widget = widgets.BoundedFloatText(value=5.0, min=1.0, max=3600.0, step=1.0, description="Refresh[s]")
plot_mode_widget = widgets.Dropdown(
    options=[("Surface wave posterior", "surface"), ("Receiver function posterior", "rf")],
    value="surface",
    description="Plot",
    layout=widgets.Layout(width="320px"),
)

main_editor = widgets.Textarea(
    value="",
    description="joint_inv",
    layout=widgets.Layout(width="100%", height="460px"),
)
recv_editor = widgets.Textarea(
    value="",
    description="recv_func.in",
    layout=widgets.Layout(width="100%", height="300px"),
)
disper_editor = widgets.Textarea(
    value="",
    description="disper.in",
    layout=widgets.Layout(width="100%", height="360px"),
)

param_widgets = {}
for key, kind, default, min_value, max_value, step, description in PARAM_SPECS:
    if kind == "bool":
        widget = widgets.Checkbox(value=bool(default), description=key, indent=False)
    elif kind == "int":
        widget = widgets.BoundedIntText(value=int(default), min=min_value, max=max_value, step=step, description=key)
    elif kind == "float":
        widget = widgets.BoundedFloatText(value=float(default), min=min_value, max=max_value, step=step, description=key)
    else:
        widget = widgets.Text(value=str(default), description=key)
    widget.layout.width = "320px"
    widget.tooltip = description
    param_widgets[key] = widget

status_widget = widgets.HTML(value="<em>No action yet.</em>")
preview_widget = widgets.Textarea(value="", description="preview", layout=widgets.Layout(width="100%", height="220px"), disabled=True)
log_tail_widget = widgets.Textarea(value="", description="log tail", layout=widgets.Layout(width="100%", height="180px"), disabled=True)
plot_status_widget = widgets.HTML(value="")
plot_image_widget = widgets.Image(format="png", layout=widgets.Layout(max_width="100%"))

load_button = widgets.Button(description="Load files", button_style="info")
sync_widgets_button = widgets.Button(description="Read editor into widgets")
apply_widgets_button = widgets.Button(description="Apply widgets to editor")
save_button = widgets.Button(description="Save edited files", button_style="success")
check_button = widgets.Button(description="Check command")
run_button = widgets.Button(description="Start subprocess", button_style="warning")
stop_run_button = widgets.Button(description="Stop subprocess")
refresh_button = widgets.Button(description="Refresh plot", button_style="info")
refresh_log_button = widgets.Button(description="Refresh log")
start_refresh_button = widgets.Button(description="Start auto refresh", button_style="success")
stop_refresh_button = widgets.Button(description="Stop auto refresh")

if "auto_refresh_stop" in globals():
    auto_refresh_stop.set()

auto_refresh_stop = threading.Event()
auto_refresh_thread = None
current_process = None
current_log_file = None
current_run_started_at = None
last_log_refresh_at = None
last_plot_refresh_at = None


def set_status(*lines):
    if not lines:
        status_widget.value = "<em>No status message.</em>"
        return
    status_widget.value = "<br>".join(html.escape(str(line)) for line in lines)


def resolve_workdir() -> Path:
    return Path(workdir_widget.value).expanduser().resolve()


def resolve_output_path(workdir: Path) -> Path:
    raw_path = Path(output_widget.value).expanduser()
    if raw_path.is_absolute():
        return raw_path.resolve()
    return (workdir / raw_path).resolve()


def resolve_workdir_file(workdir: Path, filename: str, fallback: str) -> Path:
    raw = (filename or fallback).strip().strip("'").strip('"')
    path = Path(raw).expanduser()
    if path.is_absolute():
        return path.resolve()
    return (workdir / path).resolve()


def main_values() -> dict[str, str]:
    return parse_param_text(main_editor.value)


def summary_paths_from_editor(workdir: Path) -> tuple[Path, Path]:
    values = main_values()
    recv_path = resolve_workdir_file(workdir, values.get("recv_func_in", "recv_func.in"), "recv_func.in")
    disper_path = resolve_workdir_file(workdir, values.get("disper_in", "disper.in"), "disper.in")
    return recv_path, disper_path


def set_widget_values_from_text(text: str):
    values = parse_param_text(text)
    for key, kind, default, *_ in PARAM_SPECS:
        if key in values:
            param_widgets[key].value = coerce_value(kind, values[key])
    workdir = resolve_workdir()
    recv_summary = resolve_workdir_file(workdir, values.get("recv_func_in", "recv_func.in"), "recv_func.in")
    disper_summary = resolve_workdir_file(workdir, values.get("disper_in", "disper.in"), "disper.in")
    if disper_summary.exists():
        curve_widget.max = max(1, count_blocks(disper_summary))
    if recv_summary.exists():
        trace_widget.max = max(1, count_blocks(recv_summary))


def current_updates() -> dict[str, str]:
    updates = {}
    for key, kind, *_ in PARAM_SPECS:
        updates[key] = format_value(kind, param_widgets[key].value)
    return updates


def build_command() -> tuple[list[str], Path, Path, Path]:
    workdir = resolve_workdir()
    output_path = resolve_output_path(workdir)
    binary = DEFAULT_BINARY.resolve()
    input_arg = os.path.relpath(output_path, workdir)
    command = ["mpirun", "-np", str(mpi_np_widget.value), str(binary), input_arg]
    return command, workdir, output_path, binary


def preview_saved_files(workdir: Path, output_path: Path, recv_path: Path, disper_path: Path):
    lines = []
    for label, path in [
        ("joint_inv input", output_path),
        ("receiver function summary", recv_path),
        ("dispersion summary", disper_path),
    ]:
        lines.append(f"{label}: {path} -> {'OK' if path.exists() else 'MISSING'}")
    lines.extend(["", "Current joint_inv editor preview", "-" * 32, main_editor.value])
    preview_widget.value = "\n".join(lines)


def reset_run_state():
    global current_process, current_log_file, current_run_started_at, last_plot_refresh_at
    if current_process is not None and current_process.poll() is None:
        return False
    current_process = None
    current_log_file = None
    current_run_started_at = None
    last_plot_refresh_at = None
    return True


def load_files(_=None):
    template_path = Path(template_widget.value).expanduser().resolve()
    workdir = resolve_workdir()
    if not reset_run_state():
        set_status("Cannot load a different case while joint_inv is running. Stop it first.")
        return
    if not template_path.exists():
        set_status(f"Template not found: {template_path}")
        return
    main_editor.value = template_path.read_text(encoding="utf-8")
    set_widget_values_from_text(main_editor.value)
    recv_path, disper_path = summary_paths_from_editor(workdir)
    recv_editor.value = recv_path.read_text(encoding="utf-8") if recv_path.exists() else ""
    disper_editor.value = disper_path.read_text(encoding="utf-8") if disper_path.exists() else ""
    set_status(
        f"Loaded main parameter file: {template_path}",
        f"Loaded receiver function summary: {recv_path if recv_path.exists() else 'MISSING: ' + str(recv_path)}",
        f"Loaded dispersion summary: {disper_path if disper_path.exists() else 'MISSING: ' + str(disper_path)}",
        "Edit the text areas directly, or use widgets and apply them to the main editor.",
    )
    preview_saved_files(workdir, resolve_output_path(workdir), recv_path, disper_path)


def sync_widgets_from_editor(_=None):
    set_widget_values_from_text(main_editor.value)
    set_status("Updated widgets from the current joint_inv editor text.")


def apply_widgets_to_editor(_=None):
    lines = main_editor.value.splitlines(keepends=True)
    try:
        main_editor.value = "".join(update_param_lines(lines, current_updates()))
    except KeyError as exc:
        set_status(exc)
        return
    set_status("Applied widget values to the joint_inv editor text.")


def save_edited_files(_=None):
    workdir = resolve_workdir()
    output_path = resolve_output_path(workdir)
    recv_path, disper_path = summary_paths_from_editor(workdir)
    if not reset_run_state():
        set_status("Cannot save edited files while joint_inv is running. Stop it first.")
        return
    output_path.parent.mkdir(parents=True, exist_ok=True)
    recv_path.parent.mkdir(parents=True, exist_ok=True)
    disper_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(main_editor.value, encoding="utf-8")
    recv_path.write_text(recv_editor.value, encoding="utf-8")
    disper_path.write_text(disper_editor.value, encoding="utf-8")
    set_widget_values_from_text(main_editor.value)
    set_status(
        f"Saved joint_inv input: {output_path}",
        f"Saved receiver function summary: {recv_path}",
        f"Saved dispersion summary: {disper_path}",
    )
    preview_saved_files(workdir, output_path, recv_path, disper_path)


def check_command(_=None):
    command, workdir, output_path, binary = build_command()
    recv_path, disper_path = summary_paths_from_editor(workdir)
    mpirun_path = shutil.which("mpirun")
    lines = [
        f"Workdir          : {workdir}",
        f"Template         : {Path(template_widget.value).expanduser().resolve()}",
        f"Working input    : {output_path} -> {'OK' if output_path.exists() else 'MISSING'}",
        f"recv_func summary: {recv_path} -> {'OK' if recv_path.exists() else 'MISSING'}",
        f"disper summary   : {disper_path} -> {'OK' if disper_path.exists() else 'MISSING'}",
        f"joint_inv binary : {binary} -> {'OK' if binary.exists() else 'MISSING'}",
        f"mpirun           : {mpirun_path if mpirun_path else 'MISSING'}",
        "Command",
        " ".join(command),
    ]
    if not output_path.exists():
        lines.append("Working input is not saved yet. Click `Save edited files` first.")
    set_status(*lines)


def run_subprocess(_=None):
    global current_process, current_log_file, current_run_started_at
    command, workdir, output_path, binary = build_command()
    if current_process is not None and current_process.poll() is None:
        set_status(f"joint_inv is already running. PID={current_process.pid}")
        return
    if not output_path.exists():
        set_status(f"Working input not found: {output_path}", "Save the edited files before running.")
        return
    if not binary.exists():
        set_status(f"joint_inv binary not found: {binary}")
        return
    if shutil.which("mpirun") is None:
        set_status("mpirun not found in PATH.", "Command", " ".join(command))
        return
    current_run_started_at = time.time()
    current_log_file = workdir / "joint_inv_ui.log"
    log_handle = current_log_file.open("w", encoding="utf-8")
    current_process = subprocess.Popen(
        command,
        cwd=workdir,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        text=True,
    )
    log_handle.close()
    set_status(
        "Started joint_inv asynchronously",
        " ".join(command),
        f"cwd={workdir}",
        f"PID={current_process.pid}",
        f"Log={current_log_file}",
        "Use Refresh log / Refresh plot or Start auto refresh while the process runs.",
    )
    refresh_log()


def stop_subprocess(_=None):
    global current_process
    if current_process is None or current_process.poll() is not None:
        set_status("No running joint_inv subprocess.")
        return
    current_process.terminate()
    set_status(f"Terminate requested for PID={current_process.pid}")
    refresh_log()


def read_log_tail(log_path: Path, max_lines: int = 8) -> str:
    if not log_path.exists():
        return f"Log file does not exist yet: {log_path}"
    with log_path.open("r", encoding="utf-8", errors="replace") as handle:
        lines = deque(handle, maxlen=max_lines)
    tail = "".join(lines).rstrip()
    return tail if tail else "Log file is empty."


def refresh_log(_=None):
    global last_log_refresh_at
    workdir = resolve_workdir()
    log_path = resolve_log_path(workdir)
    last_log_refresh_at = time.time()
    status = [
        process_status_text(),
        f"Auto refresh: {'running' if auto_refresh_thread is not None and auto_refresh_thread.is_alive() else 'stopped'}",
        f"Log refresh: {format_timestamp(last_log_refresh_at)}",
        f"Posterior plot refresh: {format_timestamp(last_plot_refresh_at)}",
        f"Log: {log_path} -> {'OK' if log_path.exists() else 'MISSING'}",
    ]
    if log_path.exists():
        status.append(f"Size: {log_path.stat().st_size} bytes")
    log_tail_widget.value = "\n".join(status + ["-" * 72, read_log_tail(log_path, max_lines=8)])


def process_status_text() -> str:
    if current_process is None:
        return "No joint_inv subprocess started from this notebook."
    code = current_process.poll()
    if code is None:
        return f"joint_inv is running. PID={current_process.pid}"
    return f"joint_inv exited with return code {code}."


def build_result() -> tuple[InvResult, Path]:
    workdir = resolve_workdir()
    output_path = resolve_output_path(workdir)
    if not output_path.exists():
        raise FileNotFoundError(f"Working input not found: {output_path}")
    result = InvResult(str(output_path))
    return result, workdir


def required_posterior_files(workdir: Path) -> list[Path]:
    curve = str(curve_widget.value).zfill(3)
    trace = str(trace_widget.value).zfill(3)
    files = [
        "n_layers.ppd",
        "vs_z.ppd",
        "vs_z.mean",
        "proposal.count",
    ]
    if parse_bool(main_values().get("solve_vp", "F")):
        files.extend(["vp_z.ppd", "vp_z.mean"])
    if plot_mode_widget.value == "surface":
        files.extend([
            f"syn_phase{curve}.ppd",
            f"syn_group{curve}.ppd",
            f"syn_hv{curve}.ppd",
            f"syn_ra{curve}.ppd",
            f"phase_sigma{curve}.ppd",
            f"group_sigma{curve}.ppd",
            f"hv_sigma{curve}.ppd",
            f"ra_sigma{curve}.ppd",
        ])
    else:
        files.extend([f"syn_rf{trace}.ppd", f"rf_sigma{trace}.ppd"])
    return [workdir / name for name in files]


def missing_posterior_files(workdir: Path) -> list[Path]:
    missing = []
    for path in required_posterior_files(workdir):
        if not path.exists() or path.stat().st_size == 0:
            missing.append(path)
            continue
        if current_run_started_at is not None and path.stat().st_mtime < current_run_started_at:
            missing.append(path)
    return missing


def newest_posterior_mtime(workdir: Path):
    times = [path.stat().st_mtime for path in required_posterior_files(workdir) if path.exists()]
    return max(times) if times else None


def posterior_snapshot_ready(workdir: Path) -> bool:
    return len(missing_posterior_files(workdir)) == 0


def posterior_status_message(workdir: Path) -> str:
    values = main_values()
    burn = values.get("n_burn", "?")
    corr = values.get("n_corr", "?")
    missing = missing_posterior_files(workdir)
    if not missing:
        mtime = newest_posterior_mtime(workdir)
        stamp = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(mtime)) if mtime else "unknown"
        return f"Posterior snapshot is available. Final posterior files contain post-burn samples only. Latest file update: {stamp}"
    names = ", ".join(path.name for path in missing[:5])
    if len(missing) > 5:
        names += f", ... ({len(missing)} files missing)"
    return (
        "Posterior snapshot is not complete yet. During burn-in, joint_inv does not update final posterior files "
        f"(`*.ppd`, `*.mean`, `syn_*`, `*_sigma`). Waiting for post-burn output after n_burn={burn}, n_corr={corr}. "
        f"Missing: {names}"
    )


def render_selected_figure(result: InvResult):
    if plot_mode_widget.value == "surface":
        return result.create_surface_wave_figure(curve_widget.value)
    return result.create_recv_func_figure(trace_widget.value)


def refresh_plot(_=None):
    global last_plot_refresh_at
    fig = None
    messages = []
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            result, workdir = build_result()
        requested_at = time.time()
        messages.extend([
            f"Plot refresh requested: {format_timestamp(requested_at)}",
            f"Previous successful plot refresh: {format_timestamp(last_plot_refresh_at)}",
            posterior_status_message(workdir),
        ])
        if not posterior_snapshot_ready(workdir):
            plot_status_widget.value = "<br>".join(html.escape(message) for message in messages)
            return
        with pushd(workdir), contextlib.redirect_stdout(io.StringIO()):
            fig = render_selected_figure(result)
        buffer = io.BytesIO()
        fig.savefig(buffer, format="png", bbox_inches="tight")
        plot_image_widget.value = buffer.getvalue()
        last_plot_refresh_at = requested_at
        messages.append(f"Posterior plot refreshed: {format_timestamp(last_plot_refresh_at)}")
        plot_status_widget.value = "<br>".join(html.escape(message) for message in messages)
    except Exception as exc:
        messages.extend([
            "Refresh failed. Files may still be missing or being written.",
            f"{type(exc).__name__}: {exc}",
            traceback.format_exc(limit=2),
        ])
        plot_status_widget.value = "<br>".join(html.escape(message) for message in messages)
    finally:
        if fig is not None:
            plt.close(fig)


def auto_refresh_loop():
    while not auto_refresh_stop.wait(interval_widget.value):
        refresh_log()
        refresh_plot()


def start_auto_refresh(_=None):
    global auto_refresh_thread
    if auto_refresh_thread is not None and auto_refresh_thread.is_alive():
        set_status("Auto refresh is already running.")
        return
    auto_refresh_stop.clear()
    auto_refresh_thread = threading.Thread(target=auto_refresh_loop, daemon=True)
    auto_refresh_thread.start()
    set_status(
        f"Auto refresh started: every {interval_widget.value:.1f} s",
        "Process status, log tail, and posterior plot area will update together.",
        "Posterior plots remain empty until the first post-burn snapshot exists.",
    )
    refresh_log()
    refresh_plot()


def stop_auto_refresh(_=None):
    auto_refresh_stop.set()
    set_status("Auto refresh stopped.")


load_button.on_click(load_files)
sync_widgets_button.on_click(sync_widgets_from_editor)
apply_widgets_button.on_click(apply_widgets_to_editor)
save_button.on_click(save_edited_files)
check_button.on_click(check_command)
run_button.on_click(run_subprocess)
stop_run_button.on_click(stop_subprocess)
refresh_button.on_click(refresh_plot)
refresh_log_button.on_click(refresh_log)
start_refresh_button.on_click(start_auto_refresh)
stop_refresh_button.on_click(stop_auto_refresh)
plot_mode_widget.observe(refresh_plot, names="value")

load_files()
refresh_log()
refresh_plot()


In [4]:
path_box = widgets.VBox([
    workdir_widget,
    template_widget,
    output_widget,
    widgets.HBox([mpi_np_widget, curve_widget, trace_widget]),
    widgets.HBox([plot_mode_widget, interval_widget]),
])

bool_keys = [key for key, kind, *_ in PARAM_SPECS if kind == "bool"]
num_keys = [key for key, kind, *_ in PARAM_SPECS if kind != "bool"]

number_box = widgets.GridBox(
    [param_widgets[key] for key in num_keys],
    layout=widgets.Layout(grid_template_columns="repeat(2, minmax(320px, 1fr))", grid_gap="8px 12px"),
)

bool_box = widgets.HBox([param_widgets[key] for key in bool_keys])
button_box = widgets.HBox([
    load_button,
    sync_widgets_button,
    apply_widgets_button,
    save_button,
    check_button,
    run_button,
    stop_run_button,
])
refresh_box = widgets.HBox([
    refresh_button,
    refresh_log_button,
    start_refresh_button,
    stop_refresh_button,
])

editor_tabs = widgets.Tab(children=[main_editor, recv_editor, disper_editor])
editor_tabs.set_title(0, "joint_inv")
editor_tabs.set_title(1, "recv_func.in")
editor_tabs.set_title(2, "disper.in")

ui = widgets.VBox([
    widgets.HTML("<h3>joint_inv input editor</h3>"),
    path_box,
    widgets.HTML("<h4>File editors</h4>"),
    editor_tabs,
    widgets.HTML("<h4>Main parameter widgets</h4>"),
    number_box,
    widgets.HTML("<h4>Flags</h4>"),
    bool_box,
    button_box,
    widgets.HTML("<h4>Posterior refresh controls</h4>"),
    refresh_box,
    widgets.HTML("<p>Burn-in progress is not plotted into final posterior files. Manual/auto refresh reads only post-burn snapshots.</p>"),
    widgets.HTML("<h4>Status</h4>"),
    status_widget,
    widgets.HTML("<h4>Saved file preview</h4>"),
    preview_widget,
    widgets.HTML("<h4>joint_inv log tail</h4>"),
    log_tail_widget,
    widgets.HTML("<h4>Posterior plots</h4>"),
    plot_status_widget,
    plot_image_widget,
])

display(ui)
preview_saved_files(resolve_workdir(), resolve_output_path(resolve_workdir()), *summary_paths_from_editor(resolve_workdir()))


In [5]:
# 必要に応じてボタンを使わず手動実行するためのメモです。
# 通常は UI の Check command / Start subprocess を使ってください。
#
# command, workdir, output_path, binary = build_command()
# completed = subprocess.run(command, cwd=workdir, check=False, text=True)
# completed.returncode
